# Black-Scholes Model

The Black-Scholes model is a mathematical framework for pricing derivative contracts, particularly European-style options. Developed in 1973, it revolutionized financial markets by providing a standardized method to calculate the theoretical value (Fair Value) of an option, isolating risk.

## 1. Fundamental Assumptions (Model Hypotheses)

To make the mathematics work, the model operates in an "ideal" market and makes several strong assumptions (which often do not hold in reality):

* **European Options:** The option can be exercised *only* at the expiration date (not before, as with American options).
* **Log-Normal Distribution:** The returns of the underlying asset follow a normal distribution (prices cannot be negative and follow a geometric Brownian motion).
* **Constant Volatility:** The volatility of the underlying asset is known and constant throughout the life of the option (this is the model's greatest limitation).
* **No Dividends:** The underlying security does not pay dividends during the life of the option (model extensions exist to include them).
* **Frictionless Market:** There are no transaction costs, taxes, or short-selling restrictions. The risk-free rate ($r$) is constant.

## 2. The 5 Input Variables

To price an option with Black-Scholes, the model requires exactly 5 ingredients:

1. $S$: *Spot* price (current price) of the underlying asset.
2. $K$: *Strike* price (the price at which you have the right to buy/sell).
3. $T$: Time to expiration (expressed as a fraction of a year, e.g., 6 months = 0.5).
4. $r$: *Risk-Free* interest rate (e.g., government bond yield).
5. $\sigma$: Implied volatility of the underlying asset (annualized standard deviation of returns).

## 3. Mathematical Formulas

The theoretical price of a Call option ($C$) and a Put option ($P$) is calculated using the following stochastic differential equations:

### Call Option (Right to Buy)
$$C=S \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)$$

### Put Option (Right to Sell)
$$P=K \cdot e^{-rT} \cdot N(-d_2) - S \cdot N(-d_1)$$

**Where the probability parameters $d_1$ and $d_2$ are calculated as:**

$$d_1=\frac{\ln(S/K) + (r + \frac{\sigma^2}{2})T}{\sigma \sqrt{T}}$$

$$d_2=d_1 - \sigma \sqrt{T}$$

*Note:* $N(x)$ represents the cumulative distribution function of the standard normal distribution (indicates the probability that the option expires "In The Money").

In [5]:
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, q=0, option_type='call'):
    """
    Black-Scholes option pricing model
    
    Parameters:
    S: Current stock price
    K: Strike price
    T: Time to expiration (in years)
    r: Risk-free interest rate
    sigma: Volatility (annualized standard deviation)
    q: Dividend yield (default 0)
    option_type: 'call' or 'put'
    
    Returns:
    Option price
    """
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:  # put
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
    
    return price

# Example usage
S = 100      # Stock price
K = 100      # Strike price
T = 1        # 1 year to expiration
r = 0.05     # 5% risk-free rate
sigma = 0.2  # 20% volatility
q = 0        # 0% dividend yield

call_price = black_scholes(S, K, T, r, sigma, q=q, option_type='call')
put_price = black_scholes(S, K, T, r, sigma, q=q, option_type='put')

print(f"Call option price: ${call_price:.2f}")
print(f"Put option price: ${put_price:.2f}")

Call option price: $10.45
Put option price: $5.57


## 4. Beyond Price: The "Greeks"

The real power of Black‑Scholes is not only computing the price but understanding how that price changes when an input changes. These partial derivatives are called the "Greeks":

* **Delta ($\Delta$):** How much the option price changes if the underlying ($S$) moves by 1 unit.
* **Gamma ($\Gamma$):** The rate of change of Delta (the second derivative with respect to $S$).
* **Theta ($\Theta$):** How much value the option loses each day (time decay, derivative with respect to $T$).
* **Vega ($\nu$):** How much the option price changes for a one percentage-point increase in volatility ($\sigma$).
* **Rho ($\rho$):** Sensitivity to the interest rate ($r$).

## 5. Practical Limits for Real-World Trading

In the real world, markets suffer "Black Swans" (fat tails in the distribution) and volatility is never constant — it varies by strike (the *volatility smile* phenomenon). This model is an excellent theoretical compass, but market prices will always differ slightly from the theoretical $C$ and $P$.

In [ ]:
# Calcola le "Greche" (Delta, Gamma, Theta, Vega, Rho) per call e put usando le variabili già presenti:
# S, K, T, r, sigma, q e le funzioni/moduli importati (np, norm).

def _d1_d2(S, K, T, r, sigma, q):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

def greeks(S, K, T, r, sigma, q=0):
    d1, d2 = _d1_d2(S, K, T, r, sigma, q)
    pdf_d1 = norm.pdf(d1)
    cdf_d1 = norm.cdf(d1)
    cdf_d2 = norm.cdf(d2)
    exp_qT = np.exp(-q * T)
    exp_rT = np.exp(-r * T)

    # Delta
    delta_call = exp_qT * cdf_d1
    delta_put  = exp_qT * (cdf_d1 - 1.0)

    # Gamma (stesso per call e put)
    gamma = exp_qT * pdf_d1 / (S * sigma * np.sqrt(T))

    # Vega (sensibilità per 1 punto di volatilità, es. da 0.20 a 0.21)
    vega = S * exp_qT * pdf_d1 * np.sqrt(T)

    # Theta (espresso in unità annue). Fornisco anche il valore giornaliero dividendo per 365.
    theta_call = (
        - (S * pdf_d1 * sigma * exp_qT) / (2 * np.sqrt(T))
        - r * K * exp_rT * cdf_d2
        + q * S * exp_qT * cdf_d1
    )
    theta_put = (
        - (S * pdf_d1 * sigma * exp_qT) / (2 * np.sqrt(T))
        + r * K * exp_rT * norm.cdf(-d2)
        - q * S * exp_qT * norm.cdf(-d1)
    )

    # Rho (sensibilità ai tassi, per 1 punto di tasso, es. da 0.05

# Implied Volatility (IV): The Market's "Heartbeat"

While *Historical Volatility* measures how much a security has moved in the past, **Implied Volatility (IV)** is a forward-looking measure. It represents the market's expectation of how much the underlying will fluctuate from today until the option's expiration.

In practical terms, IV is the market's "price of fear or greed."

## 1. The Mathematical Concept (Reverse Engineering)

In the classical Black‑Scholes model we input 5 variables (including historical volatility $\sigma$) to compute a theoretical price ($C$ or $P$). However, in the real world the option price ($C_{\text{market}}$) is set by market supply and demand.

Implied Volatility is the inverse problem: we feed the market price into the model and find the unique volatility value ($\sigma_{\text{imp}}$) that justifies that price:

$$C_{\text{market}} = BS(S, K, T, r, \sigma_{\text{imp}})$$

Mathematical note: There is no closed‑form algebraic solution to isolate $\sigma_{\text{imp}}$ from the Black‑Scholes equation. Computers find it using numerical root‑finding/approximation methods (trial and error), such as the **Newton‑Raphson method**.

## 2. How Traders Interpret IV

For an investment fund, IV is the compass to judge whether an option is "expensive" or "cheap":

* **High IV:** Options are costly. The market expects a large move (e.g., before earnings or major economic releases). Institutional traders often *sell* options when IV is high to collect rich premiums.
* **Low IV:** Options are cheap. The market is calm. This is when funds often *buy* options to hedge portfolios at low cost.

## 3. IV Rank and IV Percentile

Saying "Apple has an IV of 30%" means little without context. Is 30% high or low for Apple? Two relative metrics are used:

* **IV Rank:** Compares current IV to its 52‑week high and low. (A Rank of 100 means IV is at its annual maximum.)
* **IV Percentile:** The percentage of days in the past year when IV was below today's level.

## 4. The Volatility Smile (Black‑Scholes' Limitation)

Black‑Scholes assumes volatility is constant across strikes ($K$). After the 1987 crash, traders started pricing higher risk for extreme downturns.

If we plot IV for options with the same expiration across different strikes, we don't get a flat line but a curve shaped like a "smile" or a "skew." Deep out‑of‑the‑money puts often show much higher IVs than calls, since investors pay a premium to insure against severe market drops ("tail risk").


In [17]:
def implied_vol_newton(market_price, S, K, T, r, q=0.0, option_type='call',
                       initial_sigma=0.2, tol=1e-8, max_iter=100):
    """
    Calcola la volatilità implicita tramite il metodo di Newton-Raphson.
    Dipende dalla funzione `black_scholes` e da `_d1_d2` già presenti nel notebook.
    """
    sigma = float(initial_sigma)
    for i in range(max_iter):
        price = black_scholes(S, K, T, r, sigma, q=q, option_type=option_type)
        diff = price - market_price
        if abs(diff) < tol:
            return sigma

        # calcola d1 per la vega usando la funzione _d1_d2 già definita
        d1, _ = _d1_d2(S, K, T, r, sigma, q)
        vega = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(max(T, 1e-12))

        # se vega è troppo piccolo, Newton può fallire: interrompi
        if vega < 1e-12:
            break

        # aggiornamento Newton-Raphson
        sigma -= diff / vega

        # evitare valori non fisici
        if sigma <= 0 or np.isnan(sigma):
            sigma = 1e-8

    raise RuntimeError(f"Implied vol did not converge after {max_iter} iterations (last diff={diff})")


# Esempio di utilizzo con le variabili già presenti nel notebook:
implied_vol_call = implied_vol_newton(call_price, S, K, T, r, q=q, option_type='call', initial_sigma=sigma)
implied_vol_put  = implied_vol_newton(put_price,  S, K, T, r, q=q, option_type='put',  initial_sigma=sigma)

print(f"Implied vol (call) = {implied_vol_call:.6f}")
print(f"Implied vol (put)  = {implied_vol_put:.6f}")

Implied vol (call) = 0.200000
Implied vol (put)  = 0.200000
